In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
%matplotlib inline

# Lecture 22: Multivariable Methods

**Data 145, Spring 2026: Evidence and Uncertainty**

**Instructors:** Ani Adhikari, William Fithian

---
Please run all the code cells above before reading.

---
We will start by finishing what was left undone in the previous lecture: proving Hoeffding's inequality.

Then we will switch topics to pick up a basic technique for finding joint densities of transformations, ending with a beautiful application that will help you derive the $F$ density in preparation for multiple regression next week.

---
## 1. Proof of Hoeffding's Inequality

### The Inequality
Let $X_i \in [a_i, b_i]$ be independent and let $S_n = \sum_{i=1}^n X_i$. **Hoeffding's inequality** is the right tail bound:
$$
P(S_n - ES_n \ge t) ~ \le ~ \exp \left( \frac{-2t^2}{\sum_{i=1}^n (b_i - a_i)^2} \right)
$$

#### Proof:
Let $X_i \in [a_i, b_i]$ for $i = 1, 2, \ldots, n$ be independent and let $S_n$ be their sum. Then by a familiar move, for all $s > 0$
$$
\begin{align*}
P(S_n - ES_n \ge t) ~ &= ~ P(e^{s(S_n - ES_n)} \ge e^{st}) \\
&\le ~ E\left(e^{s(S_n - ES_n)}\right)e^{-st} \\
&= e^{-st} \prod_{i=1}^n E(e^{s(X_i - EX_i)})
\end{align*}
$$

To each factor $E(e^{s(X_i - EX_i)})$ we will apply ***Hoeffding's Lemma***: If $X \in [a, b]$ has mean $0$, then 
$$
E(e^{sX}) ~ \le ~ e^{s^2(b-a)^2/8}, ~~~ s > 0
$$
The lemma takes a little work to prove. We'll postpone that for now and simply apply it to our product above.

$$
P(S_n - ES_n \ge t) ~ \le ~ e^{-st} \prod_{i=1}^n e^{s^2(b_i-a_i)^2/8}, ~~~ s > 0
$$

The best bound is the minimum of these over all positive $s$. Let's find that minimum value by our usual method.

The function being minimized is 
$$
g(s) = e^{-st} \prod_{i=1}^n e^{s^2(b_i-a_i)^2/8} = e^{-st + s^2\sum_i (b_i-a_i)^2/8} = e^{-st + s^2c/8}
$$
where $c = \sum_i (b_i - a_i)^2$.
$$
\log(g(s)) = -st + s^2c/8
$$
$$
\frac{d}{ds} \log(g(s)) = -t + 2sc/8
$$
The minimizing value of $s$ is
$$
s^* = \frac{4t}{c}
$$
Plug this into $g$ to get the best bound.
$$
\begin{align*}
g(s^*) ~ &= ~ \exp\left(-4t^2/c \right) \cdot \exp \left( 16t^2/8c \right) \\
&= \exp \left( \frac{-2t^2}{c} \right) \\
&= \exp \left( \frac{-2t^2}{\sum_{i=1}^n (b_i - a_i)^2} \right)
\end{align*}
$$

---
#### Optional: Proof of Hoeffding's Lemma
For completeness, here's a proof of the lemma we used. **Studying this proof is optional. You are welcome to skip to the next section.**

**1. Use convexity:**
Any point $X \in [a, b]$ can be written as a convex combination of the endpoints $a$ and $b$:
$$
X ~ = ~ \frac{b-X}{b-a} a + \frac{X-a}{b-a} b 
$$

We have taken $s$ to be positive. By the definition of convexity of the function $x \to e^{sx}$, the graph of the function lies at or below the straight line segment joining $(a, e^{sa})$ and $(b, e^{sb})$. That is,
$$
e^{sX} ~ \le ~ \frac{b-X}{b-a} e^{sa} + \frac{X-a}{b-a} e^{sb}
$$
So
$$
E[e^{sX}] ~ \le ~ \frac{b-E[X]}{b-a} e^{sa} + \frac{E[X]-a}{b-a} e^{sb} ~ = ~ \frac{b}{b-a} e^{sa} - \frac{a}{b-a} e^{sb}
$$
because $E(X) = 0$. Since $X$ is not a constant, $a < E(X)$. Hence $a$ is negative and $b$ is positive. So $b-a = b+\vert a \vert$ and
$$
E[e^{sX}] ~ \le ~ (1-p)e^{sa} + pe^{sb} ~~~~~~~~ \text{where } p = \frac{-a}{b-a} = \frac{\vert a \vert}{b+\vert a \vert} \in [0, 1]
$$

**2. Examine the log of the bound:** Start by writing the bound in terms of $p$ and $w = s(b-a)$ to get
$$
(1-p)e^{sa} + pe^{sb} = (1-p)e^{-pw} + pe^{(1-p)w} = e^{-pw} -pe^{-pw} + pe^we^{-pw} = e^{-pw}(1 - p + pe^w)
$$

Let the function $g$ be the log of the bound, so that
$$
E[e^{sX}] \le e^{g(w)} ~~~~~~ \text{where } g(w) = -pw + \log(1 - p + pe^w)
$$

**3. Use a Taylor expansion:** Expand $g$ around $w = 0$:
$$
g(w) = g(0) + wg'(0) + \frac{w^2}{2}g^"(v) ~~~~~~ \text{for some } v \in (0, w)
$$

Now
- $g(0) = \log(1) = 0$
- $g'(w) = -p + \displaystyle \frac{1}{1 - p + pe^w} \cdot pe^w$ so $g'(0) = 0$

So
$$
g(w) = \frac{w^2}{2}g^"(v) ~~~~~~ \text{for some } v \in (0, w)
$$

**4. Bound $g(w)$:** All that remains of $g(w)$ is the term involving the second derivative. Now
$$
g^"(w) = \frac{p(1-p)e^w}{(1 - p + pe^w)^2} = \frac{pe^w \cdot q}{(pe^w + q)^2} 
= \frac{1}{4} \cdot \frac{xy}{\left( \frac{x+y}{2} \right)^2}
~~~~ \text{where } x = pe^w, ~ y = q
$$

The last factor on the right is the ratio of:
- the square of the geometric mean of $x$ and $y$
- the square of the arithmetic mean of $x$ and $y$

Since the arithmetic mean of two non-negative numbers is at least as large as their geometric mean, that last factor is at most $1$. So
$$
g^"(w) ~ \le ~ \frac{1}{4} ~~~~~ \text{for all } w
$$

**5. Get the final bound:** Plug this back into our last expression for $g(w)$ in Step 3 to get
$$
g(w) ~ \le ~ \frac{w^2}{2} \cdot \frac{1}{4} ~ = ~ \frac{w^2}{8}
$$
Finally, recall that $w = s(b-a)$. We now have
$$
E[e^{sX}] ~ \le ~ e^{g(w)} ~ \le ~ e^{\frac{w^2}{8}} ~ = ~ e^{s^2(b-a)^2 / 8}
$$

---
## 2. Change of Variable for Joint Densities

We hope you took a break after that last calculation, or that you skipped it entirely (though it's very clever and uses many different math inequalities, so we hope you'll go through it sometime). Here's a complete change of topic.

Recall from [your probability class](https://data140.org/textbook/content/chapter-16/monotone-functions/#change-of-variable-formula-for-density-monotone-function) how we used change of variable to find the density of transformations of single random variable: 

Let $X$ be a random variable with density $f_X$ and $U=g(X)$ where $g(x)$ is some differentiable, monotone function. The density $f_U$ is 
$$
f_U(u)=\frac{f_X(x)}{|\frac{d}{dx}g(x)|} \quad \text{at} \quad x=g^{-1}(u)
$$
or equivalently
$$
f_U(u)=f_X(g^{-1}(u))⋅|\frac{d}{dx}g^{-1}(u)|
$$

In two or more dimensions, we can apply concepts from multivariable calculus to find densities of tranformations.

Let $X$ and $Y$ have joint density $f_{X,Y}$. Let $g$ be any smooth, invertible function where $g:\mathbb{R}^2 → \mathbb{R}^2$. 

We will use the following notation:
$$g(X, Y)=(g_1(X,Y), g_2(X,Y))=(U,V)$$ 

Let 
\begin{align}
u&=g_1(x,y)\\
v&=g_2(x,y)
\end{align}

Assume that $g$ is invertible and let the function $g^{-1} = h = (h_1, h_2)$. Then

\begin{align}
x&=h_1(u,v)\\
y&=h_2(u,v)\\
\end{align}

Assume also that $g$ is differentiable.

The Jacobian $J$ is defined as

$$
  J =
  \left[ {\begin{array}{cc}
    \frac{d}{dx}g_1(x,y) & \frac{d}{dy}g_1(x,y) \\
    \frac{d}{dx}g_2(x,y) & \frac{d}{dy}g_2(x,y) \\
  \end{array} } \right]
$$
Its [inverse](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant#Inverse) $K$ is equal to
$$
  K =
  \left[ {\begin{array}{cc}
    \frac{d}{du}h_1(u,v) & \frac{d}{dv}h_1(u,v) \\
    \frac{d}{du}h_2(u,v) & \frac{d}{dv}h_2(u,v) \\
  \end{array} } \right]
$$

It follows that $det(J)=\displaystyle{\frac{1}{det(K)}}$.

By an extension of our 1-dimensional calculation, it can be shown that the density of $(U, V)$ is given by 

$$f_{U,V}(u,v)=\frac{f_{X,Y}(x,y)}{|det(J)|} \quad \text{at} \enspace x=h_1(u,v), y=h_2(u,v)$$

or equivalently

$$f_{U,V}(u,v) = f_{X,Y}(h_1(u,v),h_2(u,v)) \cdot |det(K)|$$

Either $J$ or $K$ can be computed and the corresponding equation can be used. In some situations, one will be easier than the other.

---
## 3. Example: Density of a Sum, Revisited

Suppose we have random variables $X$ and $Y$ with joint density $f_{X,Y}$. In Data 140, we said that the density $f_S$ of the sum $S=X+Y$ can be found by the calculation
$$f_S(s)= \int_{-∞}^ {∞}f_{X,Y}(x, s-x) \enspace dx$$

Let's establish this using our two-variable change of variable formula. Consider the transformation

\begin{align}
U &= X\\
V &= X + Y
\end{align}

We will find the joint density of $(U, V)$ and use that to find the marginal density of $V$.

To do this, we have to crank the handle of the machine we developed above.

\begin{align}
u&=g_1(x,y) = x\\
v&=g_2(x,y) = x + y\\
x&=h_1(u,v) = u\\
y&=h_2(u,v) = v - u \quad \text{since} \enspace x=u\\
\end{align}
\
$$
  J =
  \left[ {\begin{array}{cc}
    \frac{d}{dx}g_1(x,y) & \frac{d}{dy}g_1(x,y) \\
    \frac{d}{dx}g_2(x,y) & \frac{d}{dy}g_2(x,y) \\
  \end{array} } \right]
  = \left[ {\begin{array}{cc}
    1 & 0 \\
    1 & 1 \\
  \end{array} } \right]
$$\
$$ |det(J)| = 1 $$

Our general formula is

$$f_{U,V}(u,v)=\frac{f_{X,Y}(x,y)}{|det(J)|} \quad \text{at} \enspace x=h_1(u,v), y=h_2(u,v)$$

Hence
$$
f_{U,V}(u,v) ~ = ~ \frac{f_{X,Y}(u,v-u)}{1} ~ = ~ f_{X,Y}(u, v-u)
$$

We can now calculate the marginal density of $V$ by integrating out $U$:

$$ f_V(v)= \int_{-∞}^ {∞}f_{X,Y}(u, v -u) \enspace du $$

Remember that $V = X+Y$. The formula above is exactly the formula for the density of the sum provided in Data 140.

---
## 4. Example: Sum and Difference of i.i.d. Standard Normals, Revisited

Let $X$ and $Y$ be i.i.d. standard normal, and let 

\begin{align}
U &= X + Y\\
V &= X - Y
\end{align}

Inverses can be found by adding or subtracting $g_1$ and $g_2$:

\begin{align}
u&=g_1(x,y) = x + y\\
v&=g_2(x,y) = x - y\\
x&=h_1(u,v) = \frac{u + v}{2}\\
y&=h_2(u,v) = \frac{u - v}{2}\\
\end{align}
\
$$
  J =
 \left[ {\begin{array}{cc}
    1 & 1 \\
    1 & -1 \\
  \end{array} } \right]
$$\
$$ |det(J)| = 2 $$

We now have all the ingredients we need to plug into the change of variable formula. 

Recall the formula for density of a standard normal random variable: $f(x) = \frac{1}{\sqrt{2\pi}}e^{-\frac{1}{2}x^2}$. Since $X$ and $Y$ are independent, we can use this density to get the joint density $f_{X,Y}$, to get: 

\begin{align}
f_{U,V}(u,v) &= ~ \frac{f_{X,Y}(\frac{u + v}{2},\frac{u - v}{2})}{2} \\
&= ~ \frac{1}{2}\cdot\frac{1}{2\pi}\exp\{-\frac{1}{2}((\frac{u + v}{2})^2 + (\frac{u - v}{2})^2)\} \\
&= ~ \frac{1}{4\pi}\exp\{-\frac{1}{2}(\frac{u^2}{2} + \frac{v^2}{2})\} \\
&= ~ \frac{1}{4\pi} \exp(-\frac{1}{2} \cdot \frac{u^2}{2}) \cdot \exp(-\frac{1}{2} \cdot \frac{v^2}{2})
\end{align}

The joint density has factored, so $U$ and $V$ are independent. Examine each factor to see that each of $U$ and $V$ is normal.

The parameters of the normals are just the means and variances, which we know are $0$ and $2$ respectively by properties of expecation and variance.

Thus $U$ and $V$ independent normal $(0, 2)$ variables.

Do you need to check that $\displaystyle{\frac{1}{4\pi}}$ is the right constant? No, because $f_{U,V}$ is a joint density and hence *must* have the right constant factor. But you can check if you wish.

In Data 140 we proved that the sum and difference of two independent standard normal random variables were independent normal variables. But that process involved MGFs to show that the sum and difference are normal, and the multivariate normal density to establish independence. The change of variable formula allows us to find the joint density of the sum and difference without using any of that heavy machinery.

---
## 5. Example: Beta-Gamma Algebra

Let $X$ have the gamma $(r, \lambda)$ distribution and let $Y$ be gamma $(s, \lambda )$ independent of $X$. Define the transformed variables $U$ and $V$ as follows.

\begin{align}
U &= X + Y\\
V &=\frac{X}{X + Y}
\end{align}

$U$ is the sum and $V$ is $X$ as a proportion of the sum.

Inverses can be found by multiplying $g_1$ and $g_2$ together.

\begin{align}
u&=g_1(x,y) = x + y\\
v&=g_2(x,y) = \frac{x} {x + y}\\
x&=h_1(u,v) = uv\\
y&=h_2(u,v) = u - uv = u(1-v)
\end{align}

To avoid differentiating a quotient, we will use $K$ here.

$$
K =
  \left[ {\begin{array}{cc}
    \frac{d}{du}h_1(u,v) & \frac{d}{dv}h_1(u,v) \\
    \frac{d}{du}h_2(u,v) & \frac{d}{dv}h_2(u,v) \\
  \end{array} } \right]
  = 
 \left[ {\begin{array}{cc}
    v & u \\
    1 -v & -u \\
  \end{array} } \right]
$$

So
$$
|det(K)| = |-uv -u(1 - v)| = |-u| = u
$$

because $u > 0$.

Now plug into the change of variable formula. Recall the density of a $X$: $f_X(x) = \frac{\lambda^r}{\Gamma(r)}x^{r - 1}e^{-\lambda x}$ for $x > 0$.

\begin{align}
f_{U,V}(u,v) &= f_{X,Y}(h_1(u,v),h_2(u,v)) \cdot |det(K)|\\
&= f_{X,Y}(uv,u(1-v)) \cdot u
\end{align}

Since $X$ and $Y$ are independent, their joint density is the product of the two marginals.

\begin{align}
f_{U,V}(u,v) &= u \cdot \frac{\lambda^r}{\Gamma(r)}(uv)^{r - 1}e^{-\lambda uv} \cdot \frac{\lambda^s}{\Gamma(s)}(u(1 - v))^{s - 1}e^{-\lambda u(1 - v)} \\
&=\frac{\lambda^{r+s}}{\Gamma(r+s)}u^{r +s- 1}e^{-\lambda u} \cdot \frac{\Gamma(r + s)}{\Gamma(r)\Gamma(s)}v^{r - 1}(1 - v)^{s - 1}\\
&=f_{U}(u) \cdot f_{V}(v)
\end{align} 

We have derived the following result:

### Beta-Gamma Algebra 

Let $X$ have the gamma $(r, \lambda)$ distribution and let $Y$ be gamma $(s, \lambda )$ independent of $X$. 

Let $U = X+Y$ and $R = \displaystyle{\frac{X}{X+Y}}$. Then

- $U$ is gamma $(r + s, \lambda)$
- $V$ is beta $(r, s)$.
- $U$ and $V$ are independent.

You already knew that $X+Y$ is gamma. In your probability class, it was shown using the mgf. We now know the ratio $X/(X+Y)$ is a beta independent of the sum.

---
## Conclusion
What does this have to do with anything we've covered in the class? In the worksheet, you will use the beta-gamma algebra to derive the $F$ density that appears in inference for multiple regression. You will then use one-dimensional change of variable to derive the $t$ density that you have seen in the section on testing hypotheses. That too will appear in inference for multiple regression.

Multiple regression, along with the chi-squared, $t$, and $F$ distributions, are the topics of next week!